# Disney Exploration

In [ ]:
import sys
from pathlib import Path
from stampli.paths import get_enriched_path

import pandas as pd
import os
import importlib
from pyspark.sql import SparkSession
from stampli.util.runtime import bootstrap_spark_env
from pyspark.sql.functions import col, lower

import stampli.util.display
importlib.reload(stampli.util.display)
from stampli.util.display import display_scrollable_dataframe



In [ ]:
# Add src to sys.path so we can import stampli package
notebook_dir = Path("__file__").parent.resolve() if "__file__" in locals() else Path(".").resolve()
src_path = str(notebook_dir.parents[0]) # src/stampli -> src
if src_path not in sys.path:
    sys.path.append(src_path)
print(f"Added {src_path} to sys.path")

ENRICHED_PATH = str(get_enriched_path())


In [ ]:

bootstrap_spark_env()

spark = (SparkSession.builder
    .appName("DisneyExploration")
    .config("spark.executor.memory", "16g")
    .config("spark.driver.memory", "16g")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session Created (16GB)")

In [ ]:
column_order = [
    'Branch', 'Rating', 'sentiment_score', 'sentiment_label',
    'is_complaint', 'crowd_level', 'staff_sentiment', 'price_sensitivity',
    'family_sentiment', 'Reviewer_Location', 'topics', 'Review_Text', 'Year_Month', 'review_uid', 
]

# read enriched data in the order of columns

spd_disney = (spark.read
    .parquet(ENRICHED_PATH)
    .select(column_order)
)
disney_count = spd_disney.count()
print(f"Total Enriched Rows: {disney_count}")

In [ ]:
# 5. Create Pandas DataFrame (pd_disney)

print("Converting to Pandas (pd_disney)...")
pd_disney = spd_disney.toPandas()

print(f"Pandas DF Shape: {pd_disney.shape}")
display_scrollable_dataframe(pd_disney)

In [ ]:
pd_disney.columns

## Verification Blocks

In [ ]:
# Check for 'Crowded' / 'Packed' reviews in specific locations
print("Crowd Level Distribution:")
print(pd_disney.groupby('Branch')['crowd_level'].value_counts())

In [ ]:
# Keyword Check vs Extraction
KEYWORDS = ["crowd", "packed", "busy", "line", "queue", "wait", "full", "people"]

# Filter for reviews with keywords
mask = pd_disney['Review_Text'].str.contains('|'.join(KEYWORDS), case=False, na=False)
subset = pd_disney[mask]

print(f"Reviews with crowd keywords: {len(subset)}")

print("Of which have extracted crowd_level:")
print(subset['crowd_level'].value_counts(dropna=False))

# Show mis-matches (Keyword present, but crowd_level is NaN)
missed = subset[subset['crowd_level'].isna()]
if not missed.empty:
    print(f"\nPotential Misses ({len(missed)}):")
    display_scrollable_dataframe(missed[['Branch', 'Review_Text', 'summary']])

In [ ]:
# Specific Investigation: California in June
# User Question: "Is Disneyland California usually crowded in June?"

# 1. Parse Month
def get_month(ym):
    try:
        return int(ym.split('-')[1])
    except:
        return 0

pd_disney['month'] = pd_disney['Year_Month'].apply(get_month)

# 2. Filter CA + June
ca_june = pd_disney[
    (pd_disney['Branch'] == 'Disneyland_California') & 
    (pd_disney['month'] == 6)
]

print(f"Total Reviews (CA + June): {len(ca_june)}")
print("\nExtracted Crowd Levels:")
print(ca_june['crowd_level'].value_counts(dropna=False))

# 3. False Negative Check
KEYWORDS = ["crowd", "packed", "busy", "line", "queue", "wait"]
mask_keys = ca_june['Review_Text'].str.contains('|'.join(KEYWORDS), case=False, na=False)
missed_ca_june = ca_june[mask_keys & ca_june['crowd_level'].isna()]

print(f"\nPotential False Negatives (Keywords present, Label Missing): {len(missed_ca_june)}")
if not missed_ca_june.empty:
    display_scrollable_dataframe(missed_ca_june[['Review_Text', 'summary']])

In [ ]:
example_review = """
We visited Disneyland with our 9 and 7 year old children in June 2016. 
We bought a 3 day park hopper, and stayed in the nearby Fairfield Inn (also excellent). 
Disneyland was amazing, which is even more impressive given that I'm not a theme park person (the visit was for my wife and kids, 
not me) and the fact that we had ridiculously high expectations. Things that made our visit great: 
we used fastpasses extensively 
(learn how these work and use them to avoid the lines with the exception of 2 3 lines we didn't queue anywhere for more than 15 minutes) 
we booked dinners in the parks so we had reserved viewing areas for the shows (Paint the Night and World of Colour) 
Frozen at the Hyperion staying across the road so we could give the kids an hour of downtime each day 
(we went from park open to park close everyday) 
the characters were amazing, accessible, 
and very enthusiastic Goofy's kitchen General comments everything was clean and comfortable queues were enjoyable; 
they moved quickly, they had entertainment ride build up staff were accomodatingFavourite rides (DisneyLand)
 Splash Mountain Buzz Lightyear Astro Blasters Matterhorn Bobsleds Indiana Jones Jungle Cruise (awesome captain) 
 Space Mountain Big Thunder Mountain Star Wars The Adventure Continues (split jury on this one)
 Finding Nemo Pirates of the CarribeanFavourite rides (California Adventureland) 
 California Screamin' Grizzly River Run Tower of Terror (split decision) Radiator Springs Racing 
 (the fastpasses run out really quickly get one before 10am) Toy Story Midway Mania Soaring over CaliforniaOnly negatives 
 (pretty minor): no wifi in the park despite internet access being essential to planning (get the app to see real time queue times). 
 no stamps available in the park, 
 despite the emporium selling postcards and having a postbox immediately outside downtown dining is crap, 
 and has really long wait times (don't even bother trying to walk up without a reservation)Overall an amazing experience, 
 a must do family trip.	
"""

1️⃣ Theme × Sentiment Heatmap (High ROI, low effort)

* Extract 6–8 themes from cols and topics (queues, staff, food, price, weather, rides, cleanliness).
* Compute sentiment per theme × park × season.

Insight
-
“Queues drive 60% of negative sentiment in California during summer.”

Action
-
Reallocate staffing / fast-pass capacity seasonally per park.